# Train YOLO26 on the merged fire/smoke dataset

Primary training entry point. Run on the **training laptop** (RTX 3050+ recommended) after preparing the merged dataset.

YOLO26 is the current Ultralytics flagship — natively end-to-end (no NMS), DFL removed, 43% faster CPU inference, and the new MuSGD optimizer (default for v26) gives more stable convergence than AdamW.

**Pre-flight checklist:**
- [ ] CUDA torch installed: `pip install -r requirements-cuda.txt --index-url https://download.pytorch.org/whl/cu121`
- [ ] `ultralytics` is up to date (YOLO26 needs a recent release): `pip install -U ultralytics`
- [ ] Plugged into wall power, set Windows power profile to "Best Performance"
- [ ] `nvidia-smi` shows the GPU and at least ~6 GB free
- [ ] `data/merged/data.yaml` exists (run `scripts/prepare_dataset.py` first)
- [ ] At least 80 GB free disk

**Colab portability:** mount Drive, replace `ROOT` with the Drive path containing the merged dataset, install ultralytics, skip the local-environment cell.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('CUDA not available — install requirements-cuda.txt before training')

In [ ]:
DATA_YAML = ROOT / 'data' / 'merged' / 'data.yaml'
assert DATA_YAML.exists(), f'no merged dataset at {DATA_YAML} — run scripts/prepare_dataset.py first'
print(DATA_YAML.read_text())

## Stage 1 — baseline training

`yolo26s.pt` start, 80 epochs, **MuSGD** (the v26 default optimizer) + cosine LR + AMP. We let YOLO26 pick its own `lr0` since MuSGD wants a different default than AdamW.

Settings tuned for an 8 GB GPU (RTX 3050 / 4060). If you have ≥12 GB VRAM, bump `batch=16` to `batch=32` for a faster training pass. If you have <8 GB, drop to `batch=8`.

Cache is set to `'disk'` — `'ram'` deadlocks on Windows + Jupyter with multi-process workers. `workers=0` is the standard Windows + Jupyter fix.

The three callbacks below print a heartbeat every ~30s so you can see progress even if tqdm doesn't flush in Jupyter.

In [ ]:
import time
from ultralytics import YOLO

RUN_NAME = 'v26s-baseline'

# Heartbeat callbacks — print every ~30s so progress is visible in Jupyter even
# if the tqdm bar doesn't flush. Also marks epoch boundaries clearly.
_state = {'last': time.time(), 'batches': 0, 'epoch_start': None}

def on_epoch_start(trainer):
    _state['epoch_start'] = time.time()
    _state['batches'] = 0
    print(f'[epoch start] {trainer.epoch + 1}/{trainer.epochs}', flush=True)

def on_batch_end(trainer):
    _state['batches'] += 1
    now = time.time()
    if now - _state['last'] > 30:
        elapsed = now - _state['epoch_start']
        rate = _state['batches'] / max(elapsed, 1e-3)
        try:
            loss = float(trainer.loss)
        except Exception:
            loss = -1.0
        print(f"[heartbeat] epoch {trainer.epoch + 1} batch {_state['batches']} "
              f"({rate:.2f} it/s) loss={loss:.3f}", flush=True)
        _state['last'] = now

def on_epoch_end(trainer):
    elapsed = time.time() - _state['epoch_start']
    print(f"[epoch end] {trainer.epoch + 1} took {elapsed:.0f}s, "
          f"batches={_state['batches']}", flush=True)

model = YOLO('yolo26s.pt')
model.add_callback('on_train_epoch_start', on_epoch_start)
model.add_callback('on_train_batch_end', on_batch_end)
model.add_callback('on_train_epoch_end', on_epoch_end)

results = model.train(
    data=str(DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=16,           # 8 for 4 GB GPU, 32 for >=12 GB
    device=0,
    patience=20,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    amp=True,
    cache='disk',       # 'ram' deadlocks on Windows + Jupyter with multi-proc workers
    workers=0,          # standard Windows + Jupyter dataloader fix
    project=str(ROOT / 'runs' / 'firesmoke'),
    name=RUN_NAME,
    plots=True,
    # Note: we deliberately omit `optimizer=` and `lr0=` so YOLO26 uses its
    # MuSGD default. To force AdamW instead: optimizer='AdamW', lr0=0.001.
)
print('best weights:', model.trainer.best)

## Stage 2 — validate on the held-out test split

The test split was kept out of training and val. This is the only honest number for "will it work in production".

In [ ]:
best = ROOT / 'runs' / 'firesmoke' / RUN_NAME / 'weights' / 'best.pt'
from ultralytics import YOLO
test_model = YOLO(str(best))
test_metrics = test_model.val(data=str(DATA_YAML), split='test', imgsz=640, device=0)
print(test_metrics.box.map50, test_metrics.box.map)

## Stage 3 — copy `best.pt` into `models/`

This is what the demo's inference service loads.

In [ ]:
import shutil
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)
shutil.copy2(best, MODELS / 'best.pt')
print('->', MODELS / 'best.pt')

## Stage 4 (optional) — negative-class fine-tune

After the baseline is in place, collect industrial false-positive footage (welding, steam, dust, sunset, orange clothing) into `data/raw/industrial_negatives/` with **empty `.txt` label files**. Re-run `prepare_dataset.py` to merge them in, then continue training from the baseline `best.pt` at a 10× lower learning rate.

In [ ]:
# Uncomment after preparing industrial negatives:
# from ultralytics import YOLO
# m2 = YOLO(str(best))
# m2.train(
#     data=str(DATA_YAML), epochs=20, imgsz=640, batch=16, workers=0, cache='disk',
#     lr0=0.0001, optimizer='AdamW', cos_lr=True, amp=True,  # AdamW for fine-tune
#     project=str(ROOT / 'runs' / 'firesmoke'), name='v26s-neg-tuned')

## Stage 5 — export for deployment

In [ ]:
from ultralytics import YOLO
m = YOLO(str(MODELS / 'best.pt'))
m.export(format='onnx', opset=12, simplify=True)
# m.export(format='engine', half=True, device=0)  # NVIDIA edge (Jetson, etc.)
# m.export(format='openvino')                     # Intel CPU/GPU